In [1]:
!pip install openai requests pillow


In [2]:
import openai, requests, base64, os, json, time
print(f"openai: {openai.__version__}")
print("Imports OK")


openai: 2.32.0
Imports OK


# NeurIPS Computational Resources & Reproducibility Metadata

Required by **NeurIPS 2025 Paper Checklist §8**. Run to print a live hardware/version summary. Fill `[TODO]` fields before submission.

In [3]:
"""
NeurIPS 2025 Checklist S8 - Computational Resources
====================================================
Hardware
  GPU      : NVIDIA RTX A6000 (48 GB VRAM)
  CPU      : [TODO: e.g. AMD EPYC 7542 32-core]
  RAM      : [TODO: e.g. 256 GB DDR4]
  OS       : Windows 11 / Ubuntu 22.04
  Provider : Local on-premise workstation

Model & Inference
  Model      : gpt-5.5
  max_completion_tokens : 4096 per call
  Temp       : 0.0 (Zero-Shot / Sequential / LtM / ReAct / CoT)
               0.1 (Iterative)
               0.1-0.5 (Self-Consistency runs)
               0.1/0.7 (Meta-Prompting: analysis/generation)

API Calls per Video  (C = ceil(frames / 10))
  Zero-Shot        : C
  Sequential       : 5*C + 1
  Least-To-Most    : 8*C + 1
  ReAct            : C + 1
  True Iterative   : up to 8*C
  Self-Consistency : 5*C + 1
  Meta-Prompting   : 2*C + 2
  Chain-of-Thought : C + 1
  Total/video (50 frames, C=5) ~ 162 calls ~ 243 000 tokens

Total Compute (fill before submission)
  Dataset  : [TODO] videos x [TODO] avg frames
  Calls    : [TODO] x 162 = [TODO]
  Tokens   : [TODO] x 243 000 = [TODO]
  Time     : ~[TODO] hours

Reproducibility
  - Checkpoint files save progress after every video (atomic write)
  - Re-running any cell after failure resumes with zero extra API cost
  - All raw API responses saved as JSON before post-processing
  - Chunk-level saves after every 10-frame batch
"""
import subprocess, platform, datetime
print("=" * 64)
print(f"  NeurIPS Compute Summary  - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 64)
print(f"  Python  : {platform.python_version()}")
print(f"  OS      : {platform.platform()}")
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True).strip()
    for line in smi.split("\n"):
        print(f"  GPU     : {line.strip()}")
except Exception:
    print("  GPU     : nvidia-smi not available")
try:
    import openai
    print(f"  openai  : {openai.__version__}")
except Exception:
    pass
print(f"  Model   : gpt-5.5")
print(f"  Chunks  : 10 frames/chunk  |  max_completion_tokens : 4096")
print(f"  Retry   : 7 attempts, exponential back-off, cap 5 min")
print("=" * 64)


  NeurIPS Compute Summary  - 2026-04-28 18:21
  Python  : 3.10.11
  OS      : Windows-10-10.0.26100-SP0
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  openai  : 2.32.0
  Model   : gpt-5.5
  Chunks  : 10 frames/chunk  |  max_completion_tokens : 4096
  Retry   : 7 attempts, exponential back-off, cap 5 min


#ReAct
ReAct Prompting Approach
The ReAct technique follows a structured thinking cycle:

- Thought: Reasoning about what's being observed and its potential meaning
- Action: Deciding what specific aspect to focus on analyzing next
- Observation: Making detailed observations about that specific aspect
- Decision: Drawing a conclusion based on accumulated observations

Implementation Highlights

Explicit ReAct Framework:

- The prompt explicitly guides the model through the four-step cycle (Thought → Action → Observation → Decision)
- Applies this cycle to multiple aspects: people, actions, objects, spatial relationships, etc.


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk gets a full ReAct analysis independently


Evidence Chain Analysis:

- After all chunks are processed, a second pass synthesizes all analysis chunks
- Creates a coherent evidence chain that resolves any contradictions between chunks
- Explicitly establishes what crime occurred, who was involved, and the sequence of events
- Assesses confidence level in the final determination


Distinct Prompting Strategy:

- Unlike sequential or least-to-most approaches, ReAct emphasizes explicit reasoning Forces the model to justify conclusions with evidence
Creates a clear chain of reasoning that mimics human investigative processes



Benefits of ReAct for Crime Analysis
The ReAct approach is particularly well-suited for crime analysis because:

- Transparency: The reasoning and decision process is explicit and traceable
- Evidence-Based: Conclusions are directly linked to specific observations
- Methodical: Forces a structured investigation approach rather than jumping to conclusions
- Forensic-Style: Resembles how human investigators approach crime scenes

The final step, which combines all chunk analyses into a cohesive evidence chain, ensures that the entire video is considered holistically despite being processed in chunks. This produces a comprehensive analysis that identifies consistent evidence across the entire video sequence.
This implementation provides a robust framework for analyzing crime videos that explicitly shows the reasoning process behind each conclusion, making it particularly valuable for scenarios where explaining the rationale is as important as the conclusions themselves.

In [4]:
import os
import json
import base64
import requests
import time
from datetime import datetime
from collections import defaultdict


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
BATCH_SIZE     = 20   # max frames per API call
MAX_WORKERS    = 4    # parallel videos processed at once (OpenAI rate limits depend on tier; raise carefully)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return dict."""
    frames_data = {}
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode("utf-8")
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_data)}/{len(selected)} frames OK")
    return frames_data

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\GPT\REACT"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "react_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)








def _build_image_content(frames_data):
    """Convert frames_data dict into GPT vision content blocks, BATCH_SIZE at a time."""
    frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
    batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
    return batches, frame_names


def _make_payload(api_key, messages):
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "gpt-5.5",
        "messages": messages,
        "max_completion_tokens": 4096
    }
    return headers, payload


def _frames_to_content(frames_data, frame_names):
    """Build a GPT vision content list from a batch of frame names."""
    content = []
    for fname in frame_names:
        b64 = frames_data[fname]
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{b64}",
                "detail": "low"
            }
        })
        content.append({"type": "text", "text": f"[Frame: {fname}]"})
    return content


class ReActAnalyzer:
    """
    ReAct (Reasoning + Acting) crime analysis using GPT-5.5 with BATCH_SIZE frame batching.
    Thought 1 is text-only; Action 1 processes frames in batches (images); remaining steps text-only.
    """
    def __init__(self, api_key):
        self.api_key = api_key

    def _text_call(self, messages):
        headers, payload = _make_payload(self.api_key, messages)
        result = make_gpt_request_robust(headers, payload)
        return result.get("choices",[{}])[0].get("message",{}).get("content", str(result))

    def analyze_frames(self, frames_data, video_id, crime_type):
        print(f"\n  [ReAct] {video_id} | {len(frames_data)} frames ...")
        frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
        batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
        react_log = {}
        conversation = []

        # Thought 1
        print("    Thought 1: Initial reasoning ...")
        conversation.append({"role": "user", "content": (
            f"I need to analyze {len(frames_data)} security camera frames using the ReAct framework.\n\n"
            "THOUGHT 1: Before examining frames, reason about what indicators to look for "
            "to detect criminal activity in security footage. What are key behaviours and visual cues?"
        )})
        thought1 = self._text_call(conversation)
        conversation.append({"role": "assistant", "content": thought1})
        react_log["thought1"] = thought1

        # Action 1: Batched frame examination
        print(f"    Action 1 (images): {len(batches)} batches ...")
        batch_obs = []
        for idx, batch in enumerate(batches, 1):
            content = [{"type": "text", "text": f"THOUGHT 1:\n{thought1}\n\nACTION 1 - Examine Frames (Batch {idx}/{len(batches)}):"}]
            content += _frames_to_content(frames_data, batch)
            content.append({"type": "text", "text": "Based on the reasoning above, what do you observe? Report people, actions, objects, events."})
            messages = [{"role": "user", "content": content}]
            headers, payload = _make_payload(self.api_key, messages)
            result = make_gpt_request_robust(headers, payload)
            obs = result.get("choices",[{}])[0].get("message",{}).get("content", str(result))
            batch_obs.append(obs)
            print(f"      Batch {idx}: {len(obs)} chars")

        formatted = "\n\n".join(f"--- Batch {i+1} ---\n{s}" for i,s in enumerate(batch_obs))
        synth_msg = [{"role": "user", "content": f"Consolidate these observations from {len(frames_data)} frames:\n{formatted}\n\nWrite a single unified OBSERVATION."}]
        observation = self._text_call(synth_msg)
        react_log["action1_batch_obs"] = batch_obs
        react_log["observation1"] = observation

        conversation.append({"role": "user", "content": f"ACTION 1 COMPLETE - OBSERVATION:\n{observation}\n\nAll {len(frames_data)} frames across {len(batches)} batches examined."})
        conversation.append({"role": "assistant", "content": "Observation noted. I will now reason about what these findings mean."})

        for label, prompt in [
            ("thought2", "THOUGHT 2: What do the observations collectively indicate? What patterns emerge? Any contradictions or ambiguities?"),
            ("action2", "ACTION 2: Focused re-analysis — zero in on the most diagnostic evidence. What is most critical for determining crime type?"),
            ("thought3", "THOUGHT 3: Reason through crime classification. Consider each: Abuse, Arrest, Arson, Assault, Burglary, Explosion, Fighting, RoadAccidents, Robbery, Shooting, Shoplifting, Stealing, Vandalism, Normal. Which best fits?"),
            ("final_answer", "FINAL ANSWER:\nPRIMARY CLASSIFICATION: [crime type]\nCONFIDENCE LEVEL: [0-100%]\nSEVERITY: [Low/Medium/High/Critical]\nKEY EVIDENCE:\n- [point 1]\n- [point 2]\nREASONING SUMMARY: [how ReAct cycles led to conclusion]\nALTERNATIVE INTERPRETATIONS: [other explanations]\nRECOMMENDED LAW ENFORCEMENT RESPONSE: [actions]"),
        ]:
            print(f"    {label} ...")
            conversation.append({"role": "user", "content": prompt})
            response = self._text_call(conversation)
            conversation.append({"role": "assistant", "content": response})
            react_log[label] = response
            print(f"      Response: {len(response)} chars")

        return {
            "video_id": video_id, "crime_type": crime_type,
            "frames_analyzed": len(frames_data), "total_batches": len(batches),
            "batch_size": BATCH_SIZE, "prompting_technique": "REACT",
            "model": "gpt-5.5", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "react_log": react_log
        }



def make_gpt_request_robust(headers, payload, max_retries=7, base_wait=5):
    """
    Fault-tolerant OpenAI API call using requests.
    Retries on: rate limits (429), network errors, server errors (5xx).
    Exponential back-off with jitter, capped at 5 minutes.
    Returns parsed JSON on success, or error-string dict on permanent failure.
    """
    import random
    url = "https://api.openai.com/v1/chat/completions"

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)

            if response.status_code == 200:
                return response.json()

            # Rate limit
            if response.status_code == 429:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] 429 Rate limit. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Server errors – retry
            if response.status_code >= 500:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Permanent client errors (4xx except 429) – do not retry
            try:
                err = response.json()
            except Exception:
                err = response.text
            print(f"[FATAL] HTTP {response.status_code}: {err}")
            return {"error": f"HTTP_{response.status_code}", "detail": str(err)}

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as e:
            wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            wait = min(wait, 300)
            print(f"[Retry {attempt}/{max_retries}] Network error: {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s ...")
            time.sleep(wait)

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected: {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return {"error": str(e)}

    return {"error": f"All {max_retries} attempts exhausted"}





def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = ReActAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"react_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("ReAct Prompting Crime Video Analysis - ALL Frames")
    print("="*50)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        f = open(api_key_path, "r")
        api_key = f.read().strip()
        f.close()

        if not api_key:
            print("✗ Failed to load GPT API key: File is empty")
            return

        print("✓ Successfully loaded GPT API key")
        print(f"API key starts with: {api_key[:5]}...")

    except Exception as e:
        print(f"✗ Failed to load GPT API key: {str(e)}")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Frames directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    print("\n" + "="*50)
    print(f"REACT PROMPTING COMPLETE!")
    print(f"Videos processed: {len(results)}")
    print("="*50)

if __name__ == "__main__":
    run()

ReAct Prompting Crime Video Analysis - ALL Frames
Testing directory access...
Path: C:\Opeyemi\PROMPTS\FRAMES
  Exists: True
  Contains 13 items
  First few items: ['Abuse', 'Assault', 'Burglary']
Path: C:\Opeyemi\PROMPTS\RESULTS\GPT\REACT
  Exists: True
  Contains 0 items
Trying to load API key from: C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt
File exists: True
✓ Successfully loaded GPT API key
API key starts with: sk-pr...

Verifying directories:
Frames directory exists: True
Save directory exists: True

=== DISCOVERING FRAMES ===
    Root : C:\Opeyemi\PROMPTS\FRAMES
  Categories : ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
    Abuse               : 50 videos
    Assault             : 12 videos
    Burglary            : 100 videos
    Explosion           : 50 videos
    Fighting            : 50 videos
    RoadAccidents       : 150 videos
    Robbery             : 150 videos
    Shooting        